In [2]:

import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

# --- Import metrics libraries ---
from sklearn.metrics import f1_score, cohen_kappa_score, roc_auc_score, average_precision_score, mean_squared_error
from scipy.stats import pearsonr

# --- 1. Setup and Constants (Must match the training script) ---
IMAGE_DIR = "images" 
ANNOT_DIR = "annotations"
IMG_SIZE = (224, 224)
CHECKPOINT_PATH = "efficientnet_finetuned.weights.h5"

# --- 2. Re-create the Test Set (using the same logic as training) ---
print("Loading and preparing the test dataset...")
# (This section is copied from the training script to ensure the data is identical)
X, y_expr, y_valence, y_arousal = [], [], [], []
file_prefixes = sorted(list(set([f.split('_')[0] for f in os.listdir(ANNOT_DIR) if f.endswith('.npy')])))
for prefix in file_prefixes:
    try:
        img_path = os.path.join(IMAGE_DIR, f"{prefix}.jpg")
        expr_path = os.path.join(ANNOT_DIR, f"{prefix}_exp.npy")
        val_path = os.path.join(ANNOT_DIR, f"{prefix}_val.npy")
        aro_path = os.path.join(ANNOT_DIR, f"{prefix}_aro.npy")
        if not all(os.path.exists(p) for p in [img_path, expr_path, val_path, aro_path]): continue
        valence, arousal = np.load(val_path), np.load(aro_path)
        if valence == -2 or arousal == -2: continue
        img = load_img(img_path, target_size=IMG_SIZE)
        img_array = img_to_array(img)
        X.append(img_array)
        y_expr.append(np.load(expr_path))
        y_valence.append(valence)
        y_arousal.append(arousal)
    except Exception as e: print(f"Skipping {prefix} due to an error: {e}")

X = np.array(X, dtype="float32")
y_expr = np.array(y_expr, dtype="int")
y_va = np.stack((np.array(y_valence, dtype="float32"), np.array(y_arousal, dtype="float32")), axis=1)

X_train_val, X_test, y_expr_train_val, y_expr_test, y_va_train_val, y_va_test = train_test_split(
    X, y_expr, y_va, test_size=0.2, random_state=42, stratify=y_expr
)
# The test set is now created.

print(f"Test set loaded with {len(X_test)} samples.")

# --- 3. Re-build the Model Architecture ---
print("Re-building the model architecture...")
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"), tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1), tf.keras.layers.RandomContrast(0.1),
], name="data_augmentation")

inputs = Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = preprocess_input(x)
base_model = EfficientNetB0(include_top=False, weights='imagenet', input_tensor=x)
base_model.trainable = True # Set to match training configuration
for layer in base_model.layers[:-40]: layer.trainable = False

x_head = base_model.output
x_head = GlobalAveragePooling2D()(x_head)
x_head = Dense(512, activation="relu")(x_head)
x_head = Dropout(0.5)(x_head)
expr_output = Dense(8, activation="softmax", name="expression_output")(x_head)
va_output = Dense(2, activation="tanh", name="va_output")(x_head)
model = Model(inputs=inputs, outputs=[expr_output, va_output])

# --- 4. Load the Saved Weights ---
if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(f"Checkpoint file not found at '{CHECKPOINT_PATH}'. Please run the training script first.")
print(f"Loading best weights from {CHECKPOINT_PATH}...")
model.load_weights(CHECKPOINT_PATH)

# --- 5. Make Predictions on the Test Set ---
print("Making predictions on the test set...")
expr_preds_proba, va_preds = model.predict(X_test)
expr_preds_class = np.argmax(expr_preds_proba, axis=1)

# --- 6. Calculate and Display All Metrics ---

# Helper functions for custom metrics
def sign_agreement(y_true, y_pred):
    return np.mean(np.sign(y_true) == np.sign(y_pred))

def ccc(y_true, y_pred):
    mean_true, mean_pred = np.mean(y_true), np.mean(y_pred)
    var_true, var_pred = np.var(y_true), np.var(y_pred)
    cov = np.mean((y_true - mean_true) * (y_pred - mean_pred))
    return (2 * cov) / (var_true + var_pred + (mean_true - mean_pred)**2)

print("\n" + "="*50)
print("🏆 FINAL MODEL PERFORMANCE ON TEST SET 🏆")
print("="*50)

# --- Categorical Classification Metrics ---
print("\n--- Categorical Classification Metrics ---")
accuracy = np.mean(expr_preds_class == y_expr_test)
f1 = f1_score(y_expr_test, expr_preds_class, average='macro')
kappa = cohen_kappa_score(y_expr_test, expr_preds_class)
auc_roc = roc_auc_score(y_expr_test, expr_preds_proba, multi_class='ovr', average='macro')
# Note: Krippendorff's Alpha is a specialized metric, often for inter-rater reliability.
# It's not included in scikit-learn but can be calculated with libraries like 'simpledorff'.

print(f"Accuracy:         {accuracy:.4f}")
print(f"F1-Score (Macro): {f1:.4f}")
print(f"Cohen's Kappa:    {kappa:.4f}")
print(f"AUC (ROC):        {auc_roc:.4f}")

# --- Continuous Domain Metrics (Valence and Arousal) ---
print("\n--- Continuous Domain Metrics ---")
# Separate true labels and predictions for Valence and Arousal
v_true, a_true = y_va_test[:, 0], y_va_test[:, 1]
v_pred, a_pred = va_preds[:, 0], va_preds[:, 1]

# Calculate metrics for Valence
# FIXED: Calculate RMSE by taking the square root of MSE for backward compatibility
rmse_v = np.sqrt(mean_squared_error(v_true, v_pred))
corr_v = pearsonr(v_true, v_pred)[0]
sagr_v = sign_agreement(v_true, v_pred)
ccc_v = ccc(v_true, v_pred)

# Calculate metrics for Arousal
# FIXED: Calculate RMSE by taking the square root of MSE for backward compatibility
rmse_a = np.sqrt(mean_squared_error(a_true, a_pred))
corr_a = pearsonr(a_true, a_pred)[0]
sagr_a = sign_agreement(a_true, a_pred)
ccc_a = ccc(a_true, a_pred)

print("          |  Valence  |  Arousal")
print("------------------------------------")
print(f"RMSE      |  {rmse_v: <8.4f} |  {rmse_a: <8.4f}")
print(f"CORR      |  {corr_v: <8.4f} |  {corr_a: <8.4f}")
print(f"SAGR      |  {sagr_v: <8.4f} |  {sagr_a: <8.4f}")
print(f"CCC       |  {ccc_v: <8.4f} |  {ccc_a: <8.4f}")
print("="*50)



Loading and preparing the test dataset...
Test set loaded with 800 samples.
Re-building the model architecture...
Loading best weights from efficientnet_finetuned.weights.h5...
Making predictions on the test set...
25/25 ━━━━━━━━━━━━━━━━━━━━ 28s 937ms/step

🏆 FINAL MODEL PERFORMANCE ON TEST SET 🏆

--- Categorical Classification Metrics ---
Accuracy:         0.4138
F1-Score (Macro): 0.4170
Cohen's Kappa:    0.3300
AUC (ROC):        0.8156

--- Continuous Domain Metrics ---
          |  Valence  |  Arousal
------------------------------------
RMSE      |  0.4304   |  0.3779  
CORR      |  0.5059   |  0.3563  
SAGR      |  0.7350   |  0.7588  
CCC       |  0.4929   |  0.3368  
